# 01 First calls
Example notebook for stage 1. Run each cell in order.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-01"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: load your key

The key never goes in a cell. `call_llm` reads `GEMINI_API_KEY` from an environment variable, a local `.env` file (git ignores it), or, in Colab, from **Secrets** (the key icon in the left bar: add `GEMINI_API_KEY` and turn on notebook access). The cell below prints only the provider name, never the key.

In [ ]:
from kit.llm import call_llm, pick_provider, LLMError
from kit.prices import cost
import pandas as pd

PROVIDER = pick_provider()  # gemini if GEMINI_API_KEY is set, else openai_compatible, mock if KIT_OFFLINE=1
print("Provider:", PROVIDER)
if PROVIDER == "mock":
    print("Warning: this is the offline mock model. Its outputs are not real runs, so do not submit them.")

Step 3: set the prompt and settings by hand

The system prompt holds the kit's practice help pages, so every call answers over the same help centre data.

In [ ]:
# The help pages come from the kit's practice help centre (corpus/qn-*.md).
# Admin only pages (refunds, audit log, data deletion) stay out: a customer facing bot must not see them.
# About 2,000 words of pages, so about 3,000 input tokens per call: fine on the free tier.
def load_help_pages(folder=ROOT / "corpus"):
    pages = []
    for path in sorted(folder.glob("qn-*.md")):
        _, head, body = path.read_text().split("---", 2)
        meta = dict(line.split(": ", 1) for line in head.strip().splitlines())
        if meta["visibility"] != "admin":
            pages.append(f"[{meta['id']}] {meta['title']}\n{body.strip()}")
    return pages

HELP_PAGES = load_help_pages()
SYSTEM = ("You are the help assistant for Quillnote, a note app. Answer in at most 3 short steps.\n"
          "Answer only from the help pages below. Name the page id you used, like [qn-02]. "
          "If the pages do not answer the question, say so.\n\n" + "\n\n".join(HELP_PAGES))
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 200
TRUNCATE_CAP = 10  # the last call uses this tiny cap on purpose, to force a truncation
MODEL = None  # None means the provider default (see kit/llm.py)

QUESTIONS = [
    "How do I export a note as PDF?", "My notes stopped syncing", "Can I share a notebook with my team?",
    "How do I undo a deleted note?", "Where is dark mode?", "Can I search inside images?",
    "Why was I charged twice?", "How do I add a tag?", "The app crashes when I open a big note",
    "Can I use Quillnote offline?",
]
CAPS = [MAX_OUTPUT_TOKENS] * (len(QUESTIONS) - 1) + [TRUNCATE_CAP]  # one output cap per call
print(len(HELP_PAGES), "help pages in the system prompt, about", len(SYSTEM) // 4, "tokens")

Step 4: send 10 requests (the 10th with the tiny cap)

In [ ]:
results = []
for q, cap in zip(QUESTIONS, CAPS):
    r = call_llm(q, system=SYSTEM, model=MODEL, temperature=TEMPERATURE, max_output_tokens=cap)
    results.append(r)
    print(f"{q[:40]:40} cap {cap:>3} -> {r['text'][:60]!r}")

Step 5: the usage table, one row per call

In [ ]:
usage = pd.DataFrame([{
    "call": i, "question": q, "output_cap": cap, "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
    "stop_reason": r["stop_reason"], "latency_s": r["latency_s"],
    "cost_usd": cost(r["input_tokens"], r["output_tokens"], r["model"]),
} for i, (q, cap, r) in enumerate(zip(QUESTIONS, CAPS, results), start=1)])
usage.to_csv(OUTPUT / "usage.csv", index=False)
print("Model:", results[0]["model"], "| saved to", OUTPUT.name + "/usage.csv")
usage

In [ ]:
print("Total cost: $%.5f" % usage.cost_usd.sum())
print("Median latency: %.2fs" % usage.latency_s.median())
usage[["input_tokens", "output_tokens"]].describe()

Step 6: find the forced truncation in the table

In [ ]:
truncated = usage[usage.stop_reason.astype(str).str.upper().str.contains("MAX")]
print(len(truncated), "row(s) stopped by the output cap (Gemini says MAX_TOKENS, OpenAI style APIs say length):")
print("Text of the cut off answer:", repr(results[-1]["text"]))
truncated

Step 7: trigger a 400 error on purpose

In [ ]:
try:
    call_llm(QUESTIONS[0], system=SYSTEM, model=MODEL, temperature=5)  # temperature must be 0 to 2
except LLMError as e:
    print("Caught error. Status:", e.status)
    print("Message:", e.message[:300])

Step 8: see what a 429 looks like (simulated)

In [ ]:
# A real 429 means you sent too many requests. call_llm retries with longer waits, then gives up.
# This uses the mock provider so it never touches your real quota.
try:
    call_llm("SIMULATE_429", provider="mock")
except LLMError as e:
    print("Caught error. Status:", e.status, "|", e.message)
    print("429 = rate limit. Wait and retry, slow down, or check your quota.")

Step 9: next, open the 01 probe set notebook

Your five predicted weaknesses and your short README section go at the end of that notebook, so the whole stage can be read in two notebooks.